In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask3-20260725T044332Z-1-001 (2)/Subtask3-20260725T044332Z-1-001/Subtask3/SpringerSSAT-Tiny-Multilingual-test(Masked).csv
/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask3-20260725T044332Z-1-001 (2)/Subtask3-20260725T044332Z-1-001/Subtask3/SpringerSSAT-Tiny-Multilingual-train.csv
/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask3-20260725T044332Z-1-001 (2)/Subtask3-20260725T044332Z-1-001/Subtask3/SpringerSSAT-Tiny-Multilingual-val.csv
/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-001/Subtask1/MixSub-SciHigh_test_FIRE(Masked).csv
/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-001/Subtask1/MixSub-SciHigh_val_FIRE.csv
/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-001/Subtask1/MixSub-SciHigh_train_

In [2]:
# ========== CELL 1: Setup ==========
import os, sys, glob, json, time, random, subprocess, zipfile, shutil
import numpy as np
import pandas as pd
import torch
 
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
 
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
 
WORK_DIR = "/kaggle/working"
SUBMIT_DIR = f"{WORK_DIR}/submission"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(SUBMIT_DIR, exist_ok=True)

CUDA available: True
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [3]:
# ========== CELL 2: Discover all data files ==========
DATA_ROOT = "/kaggle/input"
all_files = []
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith((".csv", ".json", ".jsonl", ".tsv")):
            all_files.append(os.path.join(root, f))
 
print(f"Found {len(all_files)} data files:")
for f in all_files:
    print(" ", f)

Found 9 data files:
  /kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask3-20260725T044332Z-1-001 (2)/Subtask3-20260725T044332Z-1-001/Subtask3/SpringerSSAT-Tiny-Multilingual-test(Masked).csv
  /kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask3-20260725T044332Z-1-001 (2)/Subtask3-20260725T044332Z-1-001/Subtask3/SpringerSSAT-Tiny-Multilingual-train.csv
  /kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask3-20260725T044332Z-1-001 (2)/Subtask3-20260725T044332Z-1-001/Subtask3/SpringerSSAT-Tiny-Multilingual-val.csv
  /kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-001/Subtask1/MixSub-SciHigh_test_FIRE(Masked).csv
  /kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-001/Subtask1/MixSub-SciHigh_val_FIRE.csv
  /kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-00

In [4]:
#========== CELL 3: Peek columns of every file (READ THIS OUTPUT) ==========
def read_any(path, nrows=None):
    """Read csv/tsv/json/jsonl with automatic encoding fallback (fixes cp1252/latin-1 files)."""
    kwargs = {"nrows": nrows} if nrows else {}
    encodings = ["utf-8", "cp1252", "latin-1"]
    last_err = None
    for enc in encodings:
        try:
            if path.endswith(".csv"):
                return pd.read_csv(path, encoding=enc, **kwargs)
            elif path.endswith(".tsv"):
                return pd.read_csv(path, sep="\t", encoding=enc, **kwargs)
            elif path.endswith(".jsonl"):
                return pd.read_json(path, lines=True, encoding=enc, **kwargs)
            else:
                df = pd.read_json(path, encoding=enc)
                return df.head(nrows) if nrows else df
        except UnicodeDecodeError as e:
            last_err = e
            continue
    raise last_err
 
def peek_full(path):
    try:
        return read_any(path, nrows=3)
    except Exception:
        return None
 
for f in all_files:
    df = peek_full(f)
    print(f, "->", (list(df.columns) if df is not None else "ERROR reading file"))

/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask3-20260725T044332Z-1-001 (2)/Subtask3-20260725T044332Z-1-001/Subtask3/SpringerSSAT-Tiny-Multilingual-test(Masked).csv -> ['Abstract', 'English_Title']
/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask3-20260725T044332Z-1-001 (2)/Subtask3-20260725T044332Z-1-001/Subtask3/SpringerSSAT-Tiny-Multilingual-train.csv -> ['Abstract', 'English_Title', 'Bengali_Title']
/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask3-20260725T044332Z-1-001 (2)/Subtask3-20260725T044332Z-1-001/Subtask3/SpringerSSAT-Tiny-Multilingual-val.csv -> ['Abstract', 'English_Title', 'Bengali_Title']
/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-001/Subtask1/MixSub-SciHigh_test_FIRE(Masked).csv -> ['Filename', 'Abstract']
/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-001/Subtask1/MixSub-SciHigh_val_

In [5]:
# ========== CELL 4a: EDIT THESE before running anything below ==========
TEAM_NAME = "NexaNLP"          # <-- EDIT: no spaces, per submission rules (underscores only)
RUN_ID = "run1"                     # <-- EDIT if you submit a second run, e.g. "run2"
REGISTERED_EMAIL_LOCAL = "team.nexanlp@gmail.com" # <-- EDIT: the part BEFORE '@' in your registered email
                                     #     final zip will be named "<this>.zip"
HF_USERNAME = None                  # <-- fill in if you want the optional HF upload cell (CELL 11) to run
HF_TOKEN = None                     # <-- fill in (or set as a Kaggle secret) for HF upload
 
 
# ========== CELL 4b: CONFIG (edit only if auto-detect in CELL 5 gets it wrong) ==========
# train_file / val_file / test_file are tracked separately. test_file will be None until
# you've uploaded the official masked test CSVs.
CONFIG = {
    "task1_highlights":  {"train_file": None, "val_file": None, "test_file": None,
                           "input_col": None, "target_col": None, "id_col": None},
    "task2_title_gen":   {"train_file": None, "val_file": None, "test_file": None,
                           "input_col": None, "target_col": None, "id_col": None},
    "task3_translation": {"train_file": None, "val_file": None, "test_file": None,
                           "input_col": None, "target_col": None, "id_col": None},
}

In [6]:
 
# ========== CELL 5: Auto-detect files/columns per task (train/val/test kept separate) ==========
def auto_detect(task_key, input_keywords, target_keywords, file_keywords=None, id_keywords=("filename",)):
    cfg = CONFIG[task_key]
    candidates = all_files
    if file_keywords:
        filtered = [f for f in candidates if any(k in f.lower() for k in file_keywords)]
        if filtered:
            candidates = filtered
 
    train_candidates = [f for f in candidates if "train" in f.lower()]
    val_candidates = [f for f in candidates if "val" in f.lower() or "dev" in f.lower()]
    test_candidates = [f for f in candidates if "test" in f.lower()]
 
    def find_cols(f):
        df = peek_full(f)
        if df is None:
            return None, None, None
        cols = list(df.columns)
        in_col = next((c for c in cols if any(k in c.lower() for k in input_keywords)), None)
        tgt_col = next((c for c in cols if any(k in c.lower() for k in target_keywords)), None)
        id_col = next((c for c in cols if any(k in c.lower() for k in id_keywords)), None)
        return in_col, tgt_col, id_col
 
    for f in train_candidates:
        in_col, tgt_col, id_col = find_cols(f)
        if in_col and tgt_col and in_col != tgt_col:
            cfg["train_file"] = f
            cfg["input_col"] = in_col
            cfg["target_col"] = tgt_col
            cfg["id_col"] = id_col
            break
 
    for f in val_candidates:
        in_col, tgt_col, id_col = find_cols(f)
        if in_col and tgt_col and in_col != tgt_col:
            cfg["val_file"] = f
            break
 
    # Test files are MASKED -- they will NOT have target_col. Only require input_col.
    for f in test_candidates:
        in_col, _, id_col = find_cols(f)
        if in_col:
            cfg["test_file"] = f
            break
 
    print(f"[{task_key}] train={cfg['train_file']}")
    print(f"[{task_key}] val  ={cfg['val_file']}")
    print(f"[{task_key}] test ={cfg['test_file']}" + ("" if cfg["test_file"] else "  <-- NOT FOUND, upload official test CSV to proceed with real submission"))
    print(f"[{task_key}] input='{cfg['input_col']}', target='{cfg['target_col']}', id_col='{cfg['id_col']}'")
 
auto_detect("task1_highlights", ["abstract", "text", "article", "paper"], ["highlight"],
            file_keywords=["highlight", "task1", "subtask1"], id_keywords=("filename",))
auto_detect("task2_title_gen", ["abstract", "text", "article", "paper"], ["title"],
            file_keywords=["title", "task2", "subtask2"], id_keywords=("filename",))
auto_detect("task3_translation", ["title", "english", "en_title", "en"], ["bengali", "bangla", "bn_title", "bn"],
            file_keywords=["bengali", "bangla", "translat", "task3", "subtask3", "multilingual"], id_keywords=("filename",))
 
print(json.dumps(CONFIG, indent=2))

[task1_highlights] train=/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-001/Subtask1/MixSub-SciHigh_train_FIRE.csv
[task1_highlights] val  =/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-001/Subtask1/MixSub-SciHigh_val_FIRE.csv
[task1_highlights] test =/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask1-20260725T045022Z-1-001 (2)/Subtask1-20260725T045022Z-1-001/Subtask1/MixSub-SciHigh_test_FIRE(Masked).csv
[task1_highlights] input='Abstract', target='Highlights', id_col='Filename'
[task2_title_gen] train=/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask2-20260725T045103Z-1-001 (2)/Subtask2-20260725T045103Z-1-001/Subtask2/SpringerSSAT-Train-SciHigh.csv
[task2_title_gen] val  =/kaggle/input/datasets/atharvyarandole/sample-dataset/Subtask2-20260725T045103Z-1-001 (2)/Subtask2-20260725T045103Z-1-001/Subtask2/SpringerSSAT-Val-Sci

In [7]:

 
# ========== CELL 6: Load, clean, downsample training data ==========
MAX_TRAIN_SAMPLES = 300   # raise this (e.g. 1000-2000) for better quality if you have more time budget
MAX_VAL_SAMPLES = 50      # only used for the internal quality-check predictions, not submission
 
def load_task_split(task_key, split):  # split = "train_file" or "val_file"
    cfg = CONFIG[task_key]
    f = cfg[split]
    df = read_any(f)
    df = df[[cfg["input_col"], cfg["target_col"]]].dropna()
    df.columns = ["input_text", "target_text"]
    df["input_text"] = df["input_text"].astype(str)
    df["target_text"] = df["target_text"].astype(str)
    return df
 
def load_task_test(task_key):
    """
    Load the masked test file: input column (+ id column if present), NO target.
    FALLBACK: if no official test file was found (CONFIG[...]['test_file'] is None),
    use the val file as a stand-in "test" input instead -- we only keep its input (and
    id) column and deliberately drop the ground-truth column, so the output format is
    identical to what a real masked test file would produce. This lets you produce a
    correctly-formatted submission using only the demo_sets data, when no separate
    official test set is available to you.
    """
    cfg = CONFIG[task_key]
    f = cfg["test_file"]
    using_val_fallback = False
    if not f:
        f = cfg["val_file"]
        using_val_fallback = True
        if not f:
            return None
    df = read_any(f)
    cols_to_keep = [cfg["input_col"]]
    if cfg["id_col"] and cfg["id_col"] in df.columns:
        cols_to_keep = [cfg["id_col"]] + cols_to_keep
    df = df[cols_to_keep].dropna(subset=[cfg["input_col"]])
    rename = {cfg["input_col"]: "input_text"}
    if cfg["id_col"] and cfg["id_col"] in df.columns:
        rename[cfg["id_col"]] = "id_text"
    df = df.rename(columns=rename)
    df["input_text"] = df["input_text"].astype(str)
    if using_val_fallback:
        print(f"[{task_key}] NOTE: no official test file found -- using val file as stand-in test input ({f})")
    return df.reset_index(drop=True)
 
def downsample(df, n, seed=SEED):
    if len(df) <= n:
        return df.reset_index(drop=True)
    return df.sample(n=n, random_state=seed).reset_index(drop=True)
 
task1_train = downsample(load_task_split("task1_highlights", "train_file"), MAX_TRAIN_SAMPLES)
task1_val   = downsample(load_task_split("task1_highlights", "val_file"), MAX_VAL_SAMPLES)
task2_train = downsample(load_task_split("task2_title_gen", "train_file"), MAX_TRAIN_SAMPLES)
task2_val   = downsample(load_task_split("task2_title_gen", "val_file"), MAX_VAL_SAMPLES)
 
task1_test = load_task_test("task1_highlights")
task2_test = load_task_test("task2_title_gen")
task3_test = load_task_test("task3_translation")  # None until official test file uploaded
 
task1_train.to_csv(f"{WORK_DIR}/task1_train.csv", index=False)
task1_val.to_csv(f"{WORK_DIR}/task1_val.csv", index=False)
task2_train.to_csv(f"{WORK_DIR}/task2_train.csv", index=False)
task2_val.to_csv(f"{WORK_DIR}/task2_val.csv", index=False)
 
if task1_test is not None:
    task1_test.to_csv(f"{WORK_DIR}/task1_test.csv", index=False)
    print("task1 test:", task1_test.shape)
else:
    print("task1 test: NOT AVAILABLE YET")
 
if task2_test is not None:
    task2_test.to_csv(f"{WORK_DIR}/task2_test.csv", index=False)
    print("task2 test:", task2_test.shape)
else:
    print("task2 test: NOT AVAILABLE YET")
 
if task3_test is not None:
    task3_test.to_csv(f"{WORK_DIR}/task3_test.csv", index=False)
    print("task3 test:", task3_test.shape)
else:
    print("task3 test: NOT AVAILABLE YET")
 
print("task1 train/val:", task1_train.shape, task1_val.shape)
print("task2 train/val:", task2_train.shape, task2_val.shape)
 

task1 test: (1840, 2)
task2 test: (348, 1)
task3 test: (20, 1)
task1 train/val: (300, 2) (50, 2)
task2 train/val: (300, 2) (50, 2)


In [8]:
 #========== CELL 7: Write the per-GPU worker script ==========
WORKER_SCRIPT = r'''
import os, sys, json, argparse
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
 
# Hard fix for the torchao ImportError: peft.tuners.lora.torchao.dispatch_torchao()
# calls is_torchao_available(), which raises ImportError if an old torchao version is
# installed -- even though we never use torchao. `pip uninstall` in CELL 0 sometimes
# doesn't persist (session restarts, permission issues, etc.), so patch the function
# directly at the exact import site instead of relying on the package being absent.
import peft.import_utils as _peft_import_utils
_peft_import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao as _peft_lora_torchao
    _peft_lora_torchao.is_torchao_available = lambda: False
except Exception as _e:
    print("torchao monkeypatch note:", _e)
 
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train_csv", required=True)
    ap.add_argument("--val_csv", required=True)
    ap.add_argument("--test_csv", default=None)          # optional: masked test set (no target)
    ap.add_argument("--instruction", required=True)
    ap.add_argument("--val_output_csv", required=True)
    ap.add_argument("--test_output_csv", default=None)
    ap.add_argument("--adapter_out_dir", required=True)   # where to save the LoRA adapter
    ap.add_argument("--model_name", default="Qwen/Qwen2.5-1.5B-Instruct")
    ap.add_argument("--max_len", type=int, default=200)
    ap.add_argument("--epochs", type=float, default=1.0)
    ap.add_argument("--batch_size", type=int, default=4)
    ap.add_argument("--grad_accum", type=int, default=2)
    args = ap.parse_args()
 
    # CUDA_VISIBLE_DEVICES is set by the launcher before this process starts, so
    # "cuda:0" here always refers to whichever physical GPU was assigned to this job.
    device = "cuda:0"
 
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
 
    # Load directly onto the assigned GPU via device_map -- do NOT call .to(device)
    # afterward; transformers can partially materialize weights onto the GPU during
    # from_pretrained itself, and a later .to() call double-allocates memory.
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.float16,   # T4 = Turing, use fp16 not bf16
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    torch.cuda.empty_cache()
 
    lora_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
 
    def build_prompt(input_text):
        return f"### Instruction:\n{args.instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
 
    def tokenize_example(input_text, target_text):
        # Tokenize prompt and target SEPARATELY and concatenate the token-ID lists
        # directly -- retokenizing the merged string lets BPE merge subwords across the
        # prompt/target boundary, silently shifting the label mask and causing the model
        # to skip learning the target's first few words.
        prompt = build_prompt(input_text)
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        target_ids = tokenizer(target_text + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
        full_ids = (prompt_ids + target_ids)[:args.max_len]
        labels = ([-100] * len(prompt_ids) + target_ids)[:args.max_len]
        return {"input_ids": full_ids, "labels": labels}
 
    train_df = pd.read_csv(args.train_csv)
    val_df = pd.read_csv(args.val_csv)
 
    class SFTDataset(torch.utils.data.Dataset):
        def __init__(self, df):
            self.examples = [
                tokenize_example(str(r.input_text), str(r.target_text))
                for r in df.itertuples()
            ]
        def __len__(self):
            return len(self.examples)
        def __getitem__(self, idx):
            return self.examples[idx]
 
    train_ds = SFTDataset(train_df)
 
    def collate_fn(batch):
        max_len = max(len(b["input_ids"]) for b in batch)
        input_ids, attn_mask, labels = [], [], []
        for b in batch:
            pad_len = max_len - len(b["input_ids"])
            input_ids.append(b["input_ids"] + [tokenizer.pad_token_id] * pad_len)
            attn_mask.append([1] * len(b["input_ids"]) + [0] * pad_len)
            labels.append(b["labels"] + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attn_mask),
            "labels": torch.tensor(labels),
        }
 
    training_args = TrainingArguments(
        output_dir="/kaggle/working/ckpt_tmp",
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        num_train_epochs=args.epochs,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy="no",
        report_to=[],
        dataloader_num_workers=2,
        optim="adamw_torch",
    )
 
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        data_collator=collate_fn,
    )
    trainer.train()
 
    # ---- Save the LoRA adapter (needed for HF upload + Task3 reuse of the Task2 model) ----
    os.makedirs(args.adapter_out_dir, exist_ok=True)
    model.save_pretrained(args.adapter_out_dir)
    tokenizer.save_pretrained(args.adapter_out_dir)
    print(f"Adapter saved to {args.adapter_out_dir}")
 
    # ---- Batched generation helper ----
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"  # required for correct batched causal-LM generation
 
    def generate_batch(texts, batch_size=16, max_new_tokens=80):
        results = []
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            prompts = [build_prompt(t) for t in batch_texts]
            inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                                truncation=True, max_length=args.max_len).to(device)
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                )
            gen_only = out[:, inputs["input_ids"].shape[1]:]
            decoded = tokenizer.batch_decode(gen_only, skip_special_tokens=True)
            results.extend([d.strip() for d in decoded])
            print(f"  generated {min(i + batch_size, len(texts))}/{len(texts)}")
        return results
 
    # ---- Predictions on val set (internal quality check, not required for submission) ----
    val_df["prediction"] = generate_batch(val_df["input_text"].tolist())
    val_df.to_csv(args.val_output_csv, index=False)
    print(f"DONE: wrote {len(val_df)} val predictions to {args.val_output_csv}")
 
    # ---- Predictions on the real (masked) test set, if provided ----
    if args.test_csv and args.test_output_csv:
        test_df = pd.read_csv(args.test_csv)
        test_df["prediction"] = generate_batch(test_df["input_text"].tolist())
        test_df.to_csv(args.test_output_csv, index=False)
        print(f"DONE: wrote {len(test_df)} TEST predictions to {args.test_output_csv}")
 
if __name__ == "__main__":
    main()
'''
 
with open(f"{WORK_DIR}/train_worker.py", "w") as f:
    f.write(WORKER_SCRIPT)
 
print(f"Worker script written to {WORK_DIR}/train_worker.py")
 
 

Worker script written to /kaggle/working/train_worker.py


In [9]:
 #========== CELL 8: Launch task1 on GPU0 and task2 on GPU1 IN PARALLEL ==========
env1 = os.environ.copy()
env1["CUDA_VISIBLE_DEVICES"] = "0"
env2 = os.environ.copy()
env2["CUDA_VISIBLE_DEVICES"] = "1"
 
cmd1 = [
    sys.executable, f"{WORK_DIR}/train_worker.py",
    "--train_csv", f"{WORK_DIR}/task1_train.csv",
    "--val_csv", f"{WORK_DIR}/task1_val.csv",
    "--instruction", "Write a concise research highlight (2-3 sentences) summarizing the key contribution of this abstract.",
    "--val_output_csv", f"{WORK_DIR}/task1_val_preds.csv",
    "--adapter_out_dir", f"{WORK_DIR}/adapter_task1",
] + (["--test_csv", f"{WORK_DIR}/task1_test.csv", "--test_output_csv", f"{WORK_DIR}/task1_test_preds_raw.csv"]
     if task1_test is not None else [])
 
cmd2 = [
    sys.executable, f"{WORK_DIR}/train_worker.py",
    "--train_csv", f"{WORK_DIR}/task2_train.csv",
    "--val_csv", f"{WORK_DIR}/task2_val.csv",
    "--instruction", "Generate a concise, informative title for this research abstract.",
    "--val_output_csv", f"{WORK_DIR}/task2_val_preds.csv",
    "--adapter_out_dir", f"{WORK_DIR}/adapter_task2",
] + (["--test_csv", f"{WORK_DIR}/task2_test.csv", "--test_output_csv", f"{WORK_DIR}/task2_test_preds_raw.csv"]
     if task2_test is not None else [])
 
print("Launching task1 (GPU0) and task2 (GPU1) in parallel...")
t0 = time.time()
 
log1 = open(f"{WORK_DIR}/task1_log.txt", "w")
log2 = open(f"{WORK_DIR}/task2_log.txt", "w")
 
p1 = subprocess.Popen(cmd1, env=env1, stdout=log1, stderr=subprocess.STDOUT)
p2 = subprocess.Popen(cmd2, env=env2, stdout=log2, stderr=subprocess.STDOUT)
 
while p1.poll() is None or p2.poll() is None:
    time.sleep(15)
    elapsed = time.time() - t0
    print(f"[{elapsed:.0f}s] task1 running={p1.poll() is None}  task2 running={p2.poll() is None}")
 
log1.close(); log2.close()
print(f"Both jobs finished in {time.time() - t0:.0f}s")
print("task1 return code:", p1.returncode, "| task2 return code:", p2.returncode)
 
if p1.returncode != 0:
    print("\n--- task1 log (tail) ---")
    print("\n".join(open(f"{WORK_DIR}/task1_log.txt").read().splitlines()[-40:]))
if p2.returncode != 0:
    print("\n--- task2 log (tail) ---")
    print("\n".join(open(f"{WORK_DIR}/task2_log.txt").read().splitlines()[-40:]))
 
 

Launching task1 (GPU0) and task2 (GPU1) in parallel...
[15s] task1 running=True  task2 running=True
[30s] task1 running=True  task2 running=True
[45s] task1 running=True  task2 running=True
[60s] task1 running=True  task2 running=True
[75s] task1 running=True  task2 running=True
[90s] task1 running=True  task2 running=True
[105s] task1 running=True  task2 running=True
[120s] task1 running=True  task2 running=True
[135s] task1 running=True  task2 running=True
[150s] task1 running=True  task2 running=True
[165s] task1 running=True  task2 running=True
[180s] task1 running=True  task2 running=True
[195s] task1 running=True  task2 running=True
[210s] task1 running=True  task2 running=True
[225s] task1 running=True  task2 running=True
[240s] task1 running=True  task2 running=False
[255s] task1 running=True  task2 running=False
[270s] task1 running=True  task2 running=False
[285s] task1 running=True  task2 running=False
[300s] task1 running=True  task2 running=False
[315s] task1 running=True 

In [10]:

# ========== CELL 9: Task 3 -- predict English title, then translate to Bengali ==========
# The Task3 test set masks BOTH English and Bengali titles, so this is really two steps:
#   (1) predict the English title from the abstract -- reuse the Task2 adapter, since
#       SpringerSSAT-Tiny-Multilingual is a subset of the same SpringerSSAT corpus Task2
#       trains on, so the Task2 model already knows this exact abstract->title mapping.
#   (2) translate that predicted English title to Bengali with a pretrained MT model
#       (facebook/nllb-200-distilled-600M) -- inference only, no fine-tuning needed.
from transformers import AutoTokenizer as _AT, AutoModelForCausalLM as _ACLM, AutoModelForSeq2SeqLM
from peft import PeftModel
import gc
 
# Clean up any GPU memory left over from a PREVIOUS failed run of this cell. If CELL 9
# crashed partway through last time (e.g. on the torchao ImportError), whatever models
# it had already loaded onto GPU0 (base Qwen model, NLLB, etc.) are still resident in
# this kernel's memory -- re-running the cell then tries to load fresh copies on top of
# that leftover memory, causing an OOM that has nothing to do with the current attempt.
for _name in ["nllb_model", "nllb_tokenizer", "t2_model", "t2_tok"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
 
# Same torchao monkeypatch as in train_worker.py -- CELL 9 runs in the main notebook
# kernel, a SEPARATE process from the subprocess worker script, so that earlier patch
# doesn't carry over here. PeftModel.from_pretrained() hits the same broken torchao
# version check, so patch it again in this process before calling it.
import peft.import_utils as _peft_import_utils
_peft_import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao as _peft_lora_torchao
    _peft_lora_torchao.is_torchao_available = lambda: False
except Exception as _e:
    print("torchao monkeypatch note:", _e)
 
task3_device = "cuda:0" if torch.cuda.is_available() else "cpu"
 
def load_task2_model_for_inference():
    base = _ACLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        torch_dtype=torch.float16,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
    tok = _AT.from_pretrained(f"{WORK_DIR}/adapter_task2")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    model = PeftModel.from_pretrained(base, f"{WORK_DIR}/adapter_task2")
    model.eval()
    return model, tok
 
def generate_titles(model, tok, texts, instruction, max_len=200, batch_size=16, max_new_tokens=80):
    def build_prompt(t):
        return f"### Instruction:\n{instruction}\n\n### Input:\n{t}\n\n### Response:\n"
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        prompts = [build_prompt(t) for t in batch]
        inputs = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(task3_device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        gen_only = out[:, inputs["input_ids"].shape[1]:]
        results.extend([d.strip() for d in tok.batch_decode(gen_only, skip_special_tokens=True)])
    return results
 
nllb_name = "facebook/nllb-200-distilled-600M"
nllb_tokenizer = _AT.from_pretrained(nllb_name, src_lang="eng_Latn")
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_name).to(task3_device)
 
def translate_en_to_bn(texts, batch_size=16, max_new_tokens=80):
    results = []
    nllb_tokenizer.src_lang = "eng_Latn"
    bn_id = nllb_tokenizer.convert_tokens_to_ids("ben_Beng")
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = nllb_tokenizer(batch, return_tensors="pt", padding=True,
                                 truncation=True, max_length=64).to(task3_device)
        out = nllb_model.generate(**inputs, forced_bos_token_id=bn_id, max_new_tokens=max_new_tokens)
        results.extend(nllb_tokenizer.batch_decode(out, skip_special_tokens=True))
    return results
 
# --- Internal quality check on val set (has ground truth) ---
task3_cfg = CONFIG["task3_translation"]
if task3_cfg["val_file"]:
    task3_val = downsample(load_task_split("task3_translation", "val_file"), MAX_VAL_SAMPLES)
    t2_model, t2_tok = load_task2_model_for_inference()
    task3_val["predicted_title_english"] = generate_titles(
        t2_model, t2_tok, task3_val["input_text"].tolist(),  # NOTE: input_text here is the Abstract
        "Generate a concise, informative title for this research abstract.")
    del t2_model; torch.cuda.empty_cache()
    task3_val["predicted_title_bengali"] = translate_en_to_bn(task3_val["predicted_title_english"].tolist())
    task3_val.to_csv(f"{WORK_DIR}/task3_val_preds.csv", index=False)
    print("task3 val predictions saved:", task3_val.shape)
else:
    print("task3_translation val_file not found -- check CONFIG.")
 
# --- Real test predictions, if the masked test file is available ---
if task3_test is not None:
    t2_model, t2_tok = load_task2_model_for_inference()
    task3_test["predicted_title_english"] = generate_titles(
        t2_model, t2_tok, task3_test["input_text"].tolist(),
        "Generate a concise, informative title for this research abstract.")
    del t2_model; torch.cuda.empty_cache()
    task3_test["predicted_title_bengali"] = translate_en_to_bn(task3_test["predicted_title_english"].tolist())
    task3_test.to_csv(f"{WORK_DIR}/task3_test_preds_raw.csv", index=False)
    print("task3 TEST predictions saved:", task3_test.shape)
else:
    print("task3 test file not available yet -- upload the official masked test CSV and rerun from CELL 5.")
 

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

... (5 outputs left out)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


task3 val predictions saved: (20, 4)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

task3 TEST predictions saved: (20, 3)


In [11]:

# ========== CELL 10: Build exact submission CSVs (per the FIRE 2026 format spec) ==========
# Task1: Filename, Abstract, Predicted_Highlights
# Task2: Abstract, Predicted_Title
# Task3: Abstract, Predicted_Title_English, Predicted_Title_Bengali
# Filenames: <TEAM_NAME>_Task<N>_<RUN_ID>.csv

def finalize_submission():
    made_any = False

    # --- Task 1 ---
    p = f"{WORK_DIR}/task1_test_preds_raw.csv"
    if os.path.exists(p):
        df = pd.read_csv(p)
        out = pd.DataFrame({
            "Filename": df["id_text"] if "id_text" in df.columns else range(1, len(df) + 1),
            "Abstract": df["input_text"],
            "Predicted_Highlights": df["prediction"],
        })
        out_path = f"{SUBMIT_DIR}/{TEAM_NAME}_Task1_{RUN_ID}.csv"
        out.to_csv(out_path, index=False)
        print("Wrote", out_path, out.shape)
        made_any = True
    else:
        print("Task1 test predictions not found yet -- run CELL 8 with task1_test available first.")

    # --- Task 2 ---
    p = f"{WORK_DIR}/task2_test_preds_raw.csv"
    if os.path.exists(p):
        df = pd.read_csv(p)
        out = pd.DataFrame({
            "Abstract": df["input_text"],
            "Predicted_Title": df["prediction"],
        })
        out_path = f"{SUBMIT_DIR}/{TEAM_NAME}_Task2_{RUN_ID}.csv"
        out.to_csv(out_path, index=False)
        print("Wrote", out_path, out.shape)
        made_any = True
    else:
        print("Task2 test predictions not found yet -- run CELL 8 with task2_test available first.")

    # --- Task 3 ---
    p = f"{WORK_DIR}/task3_test_preds_raw.csv"
    if os.path.exists(p):
        df = pd.read_csv(p)
        out = pd.DataFrame({
            "Abstract": df["input_text"],
            "Predicted_Title_English": df["predicted_title_english"],
            "Predicted_Title_Bengali": df["predicted_title_bengali"],
        })
        out_path = f"{SUBMIT_DIR}/{TEAM_NAME}_Task3_{RUN_ID}.csv"
        out.to_csv(out_path, index=False)
        print("Wrote", out_path, out.shape)
        made_any = True
    else:
        print("Task3 test predictions not found yet -- run CELL 9 with task3_test available first.")

    return made_any

finalize_submission()




Wrote /kaggle/working/submission/NexaNLP_Task1_run1.csv (1840, 3)
Wrote /kaggle/working/submission/NexaNLP_Task2_run1.csv (348, 2)
Wrote /kaggle/working/submission/NexaNLP_Task3_run1.csv (20, 3)


True

In [13]:

# ========== CELL 11 (OPTIONAL): Push trained adapters to Hugging Face Hub ==========
# The submission requires a Hugging Face model link. Fill in HF_USERNAME and HF_TOKEN in
# CELL 4a, then run this cell. It pushes the small LoRA adapters (not full model weights).
if HF_USERNAME and HF_TOKEN:
    from huggingface_hub import HfApi, login
    login(token=HF_TOKEN)
    api = HfApi()
    hf_links = {}
    for task_key, adapter_dir in [("task1_highlights", f"{WORK_DIR}/adapter_task1"),
                                   ("task2_title_gen", f"{WORK_DIR}/adapter_task2")]:
        if os.path.isdir(adapter_dir):
            repo_id = f"{HF_USERNAME}/fire2026-{TEAM_NAME}-{task_key}"
            api.create_repo(repo_id=repo_id, exist_ok=True, token=HF_TOKEN)
            api.upload_folder(folder_path=adapter_dir, repo_id=repo_id, token=HF_TOKEN)
            hf_links[task_key] = f"https://huggingface.co/{repo_id}"
            print(f"Uploaded {task_key} adapter -> {hf_links[task_key]}")
        else:
            print(f"No adapter directory found for {task_key} at {adapter_dir}")

    with open(f"{SUBMIT_DIR}/huggingface_links.txt", "w") as f:
        for k, v in hf_links.items():
            f.write(f"{k}: {v}\n")
    print("Saved huggingface_links.txt inside the submission folder")
else:
    print("Skipping HF upload -- set HF_USERNAME and HF_TOKEN in CELL 4a to enable this.")




Skipping HF upload -- set HF_USERNAME and HF_TOKEN in CELL 4a to enable this.


In [12]:

# ========== CELL 12: Package everything into the final submission zip ==========
zip_path = f"{WORK_DIR}/{REGISTERED_EMAIL_LOCAL}.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(SUBMIT_DIR):
        zf.write(os.path.join(SUBMIT_DIR, fname), arcname=fname)

print(f"Final submission zip: {zip_path}")
print("Contents:")
with zipfile.ZipFile(zip_path) as zf:
    for n in zf.namelist():
        print(" ", n)
print("\nDownload this zip from the Kaggle Output panel and email it to tohidarehman.it@jadavpuruniversity.in")

Final submission zip: /kaggle/working/team.nexanlp@gmail.com.zip
Contents:
  NexaNLP_Task2_run1.csv
  NexaNLP_Task1_run1.csv
  NexaNLP_Task3_run1.csv

Download this zip from the Kaggle Output panel and email it to tohidarehman.it@jadavpuruniversity.in
